<a href="https://colab.research.google.com/github/Justine-paras/dasmarinas-heat-mitigation/blob/main/Dasma_Heat_Mitigation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
i !pip install -q geemap
import ee, geemap

ee.Authenticate()
ee.Initialize(project='dasma-heat')  # your Cloud project ID

import json
with open('dasmarinas_barangays.geojson') as f:
    gj = json.load(f)
for feat in gj['features']:
    feat['id'] = str(feat['id'])

brgy = ee.FeatureCollection(gj)
city = brgy.geometry().dissolve()
print('Barangays loaded:', brgy.size().getInfo())

def to_lst(img):
    qa = img.select('QA_PIXEL')
    clear = (qa.bitwiseAnd(1 << 1).eq(0)          # dilated cloud
             .And(qa.bitwiseAnd(1 << 3).eq(0))    # cloud
             .And(qa.bitwiseAnd(1 << 4).eq(0)))   # cloud shadow
    far_from_cloud = img.select('ST_CDIST').multiply(0.01).gt(0.5)  # km
    lst = (img.select('ST_B10').multiply(0.00341802).add(149.0)
              .subtract(273.15).rename('LST'))
    return lst.updateMask(clear).updateMask(far_from_cloud)

def dry_season(year):
    col = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
           .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
           .filterBounds(city)
           .filterDate(f'{year}-02-01', f'{year}-05-31')
           .filter(ee.Filter.lt('CLOUD_COVER', 60)))
    print(year, 'scenes:', col.size().getInfo())
    return col.map(to_lst).median().clip(city)

lst = {y: dry_season(y) for y in [2016, 2020, 2024]}

m = geemap.Map(basemap='HYBRID')
m.centerObject(city, 12)
m.addLayer(lst[2024], {'min': 26, 'max': 40,
           'palette': ['#ffffcc', '#fed976', '#fd8d3c', '#e31a1c', '#800026']}, 'LST 2024')
m.addLayer(brgy.style(color='000000', fillColor='00000000', width=1), {}, 'Barangays')
m

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 38.9 MB/s eta 0:00:00


In [ ]:
for y in [2016, 2020, 2024]:
    col = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
           .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
           .filterBounds(city)
           .filterDate(f'{y}-02-01', f'{y}-05-31')
           .filter(ee.Filter.lt('CLOUD_COVER', 60)))
    dates = col.aggregate_array('DATE_ACQUIRED').getInfo()
    print(y, sorted(dates))

In [ ]:
total = ee.Image(1).clip(city).reduceRegion(
    ee.Reducer.count(), city, 100).get('constant')

def with_stats(img):
    lst = to_lst(img)
    s = lst.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.count(), '', True),
                         city, 100)
    return lst.set({'city_mean': s.get('LST_mean'),
                    'clear': ee.Number(s.get('LST_count')).divide(total),
                    'date': img.get('DATE_ACQUIRED')})

def dry_season_anom(year, min_clear=0.5, start = '02-01'):
    col = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
           .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
           .filterBounds(city)
           .filterDate(f'{year}-{start}', f'{year}-05-31')
           .filter(ee.Filter.lt('CLOUD_COVER', 60))
           .map(with_stats)
           .filter(ee.Filter.gt('clear', min_clear)))
    info = col.aggregate_array('date').zip(col.aggregate_array('clear')).zip(
           col.aggregate_array('city_mean')).getInfo()
    for (d, c), m in info:
        print(year, d, f'clear {c*100:.0f}%', f'city mean {m:.1f} °C')
    anom = col.map(lambda i: i.subtract(ee.Number(i.get('city_mean')))
                              .rename('LST_anom'))
    return anom.median().clip(city)

anom = {y: dry_season_anom(y) for y in [2016, 2020, 2024]}

In [ ]:
for y in [2019, 2020, 2021]:
    print('----', y)
    dry_season_anom(y, start='01-01')

print('---- 2016 widened')
dry_season_anom(2016, start='01-01')

In [ ]:
import ee
import json

# ── 1. Initialize (reuse your session if already authenticated earlier) ──
# If Earth Engine is already initialized in this notebook, you can skip these two lines.
ee.Authenticate()
ee.Initialize(project='dasma-heat')  # your actual project ID

# ── 2. Load real Dasmariñas barangay boundaries ──────────────────────
with open('/content/sample_data/dasmarinas_barangays.geojson') as f:
    gj = json.load(f)
for feat in gj['features']:
    feat['id'] = str(feat['id'])  # EE requires string ids

brgy = ee.FeatureCollection(gj)
city = brgy.geometry().dissolve()
print('Barangays loaded:', brgy.size().getInfo())

# ── 3. Scan function: dry-season only, full cloud/cirrus/shadow masking ──
def scan_years(city, year_start, year_end, month_start=1, month_end=5, min_clear=50):
    """
    Scans each year in [year_start, year_end], restricted to month_start-month_end
    (default: dry season, Jan-May) for Landsat 8/9 scenes over `city`.
    Masks out cloud, cloud shadow, cirrus, AND dilated cloud pixels before
    computing both the clear-% check and the LST mean, so contaminated
    pixels never sneak into the average.
    """
    l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    col = l8.merge(l9)

    results = []
    for year in range(year_start, year_end + 1):
        start = f'{year}-{month_start:02d}-01'
        end = f'{year}-{month_end:02d}-28'

        scenes = (col.filterBounds(city)
                     .filterDate(start, end)
                     .filterMetadata('CLOUD_COVER', 'less_than', 70))

        n = scenes.size().getInfo()
        if n == 0:
            print(f'{year}: no candidate scenes')
            continue

        scene_list = scenes.toList(n)
        print(f'{year}: checking {n} candidate scene(s)...')

        for i in range(n):
            img = ee.Image(scene_list.get(i))
            date = img.date().format('YYYY-MM-dd').getInfo()

            # QA_PIXEL bits: 1=dilated cloud, 2=cirrus, 3=cloud, 4=cloud shadow
            qa = img.select('QA_PIXEL')
            clear_mask = (qa.bitwiseAnd(1 << 1).eq(0)
                          .And(qa.bitwiseAnd(1 << 2).eq(0))
                          .And(qa.bitwiseAnd(1 << 3).eq(0))
                          .And(qa.bitwiseAnd(1 << 4).eq(0)))

            clear_frac = clear_mask.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=city,
                scale=100,
                maxPixels=1e9
            ).get('QA_PIXEL')

            clear_pct = ee.Number(clear_frac).multiply(100).getInfo()

            if clear_pct is not None and clear_pct >= min_clear:
                lst_img = (img.select('ST_B10')
                              .multiply(0.00341802).add(149.0).subtract(273.15)
                              .updateMask(clear_mask))  # only average clear pixels

                lst = lst_img.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=city,
                    scale=100,
                    maxPixels=1e9
                ).get('ST_B10')
                mean_c = ee.Number(lst).getInfo()

                results.append({
                    'year': year,
                    'date': date,
                    'month': int(date.split('-')[1]),
                    'clear_pct': round(clear_pct, 1),
                    'mean_c': round(mean_c, 1) if mean_c is not None else None
                })

    results.sort(key=lambda r: -r['clear_pct'])
    return results

# ── 4. Run the sweep: Landsat 8/9, 2013–2024, dry season (Jan–May) only ──
all_candidates = scan_years(city, 2013, 2024, month_start=1, month_end=5, min_clear=50)

print('\n=== USABLE DRY-SEASON SCENES (sorted by clearest first) ===')
for r in all_candidates:
    print(f"{r['date']}  |  clear {r['clear_pct']:>5.1f}%  |  mean {r['mean_c']:>5.1f} °C")

# ── 5. Best pick nearest each target year, dry-season only ────────────
def best_near(candidates, target_year, window=2):
    nearby = [r for r in candidates if abs(r['year'] - target_year) <= window]
    if not nearby:
        return None
    # prefer highest clear %, tie-break by closeness to the target year
    nearby.sort(key=lambda r: (-r['clear_pct'], abs(r['year'] - target_year)))
    return nearby[0]

print('\n=== BEST DRY-SEASON PICK PER TARGET PERIOD ===')
for target in [2016, 2020, 2024]:
    pick = best_near(all_candidates, target, window=2)
    if pick:
        print(f"~{target}: use {pick['date']}  (clear {pick['clear_pct']}%, {pick['mean_c']}°C)")
    else:
        print(f"~{target}: NO usable dry-season scene within ±2 years — widen the window")

In [ ]:
import ee
import pandas as pd

# ── 1. Function: get per-barangay mean LST for one specific date ──────
def get_barangay_lst(date_str, brgy, label):
    img = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
             .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
             .filterBounds(brgy)
             .filterDate(date_str, ee.Date(date_str).advance(1, 'day'))
             .first())

    qa = img.select('QA_PIXEL')
    clear_mask = (qa.bitwiseAnd(1 << 1).eq(0)
                  .And(qa.bitwiseAnd(1 << 2).eq(0))
                  .And(qa.bitwiseAnd(1 << 3).eq(0))
                  .And(qa.bitwiseAnd(1 << 4).eq(0)))

    lst = (img.select('ST_B10')
              .multiply(0.00341802).add(149.0).subtract(273.15)
              .updateMask(clear_mask)
              .rename('lst'))

    # reduceRegions computes the mean separately for EACH barangay polygon
    per_brgy = lst.reduceRegions(
        collection=brgy,
        reducer=ee.Reducer.mean(),
        scale=100
    )

    # pull the results down from Earth Engine into a normal Python table
    feats = per_brgy.getInfo()['features']
    rows = []
    for f in feats:
        props = f['properties']
        rows.append({
            'barangay': props.get('adm4_en'),   # barangay name field from your geojson
            f'{label}_lst': props.get('mean')
        })
    return pd.DataFrame(rows)

# ── 2. Run it for your three confirmed clean dates ────────────────────
df_2016 = get_barangay_lst('2016-02-13', brgy, '2016')
df_2019 = get_barangay_lst('2019-02-21', brgy, '2019')  # your "~2020" stand-in
df_2024 = get_barangay_lst('2024-02-11', brgy, '2024')

# ── 3. Merge into one table: one row per barangay, one column per year ─
result = df_2016.merge(df_2019, on='barangay').merge(df_2024, on='barangay')
result['change_2016_2024'] = result['2024_lst'] - result['2016_lst']
result = result.sort_values('change_2016_2024', ascending=False)

print(result.to_string(index=False))

# ── 4. Save it so you don't have to rerun this every time ─────────────
result.to_csv('barangay_lst_by_year.csv', index=False)
print('\nSaved to barangay_lst_by_year.csv')

# New section